In [29]:
import pandas as pd
from sklearn.neighbors import BallTree

EARTH_RADIUS_KM = 6371.0088

speed = pd.read_csv("outputs/speedtests_clean.csv")
towers = pd.read_csv("outputs/towers_clean.csv")

In [33]:
speed.columns

Index(['date', 'latitude', 'longitude', 'avg_d_mbps', 'avg_u_mbps',
       'avg_lat_ms', 'tests', 'block_number', 'city', 'area', 'typeOfArea',
       'region', 'digital_elevation_model', 'last_modified_date', 'year',
       'quarter'],
      dtype='object')

In [26]:
from math import radians

speed_geo_long_in_radians = [radians(c) for c in speed['longitude']]
type(speed_geo_long_in_radians)

list

In [27]:
speed_geo_lat_in_radians = [radians(c) for c in speed['latitude']]
type(speed_geo_lat_in_radians)

list

In [30]:
import numpy as np

def to_radians(df, lat_col, lon_col):
    return np.deg2rad(df[[lat_col, lon_col]].to_numpy(dtype=float))

speed_coords = to_radians(speed, "latitude", "longitude")

In [25]:
type(speed_coords)

numpy.ndarray

In [34]:
towers.columns

Index(['Site ID', 'Region ID', 'RAT Type', 'RAT SubType', 'Latitude',
       'Longitude', 'Bands', 'Channels', 'First Seen', 'Last Seen', 'Visible',
       'Tower Type', 'Operator'],
      dtype='object')

In [35]:
tower_coords = to_radians(towers, "Latitude", "Longitude")

In [36]:
tower_coords

array([[0.45631809, 0.88152393],
       [0.45775624, 0.88187648],
       [0.45833744, 0.88318896],
       ...,
       [0.45630063, 0.88130053],
       [0.45506144, 0.88189045],
       [0.45652578, 0.88151695]])

In [37]:
tree = BallTree(tower_coords, metric="haversine")

In [38]:
#Find the neearest (1) tower to the speedtest record, index of the tower, and the distence (how far) it is in radiance
dist_rad, ind = tree.query(speed_coords, k=1)
speed["nearest_tower_distance_km"] = dist_rad[:, 0] * EARTH_RADIUS_KM

In [46]:
speed.tail(10)

,date,latitude,longitude,avg_d_mbps,avg_u_mbps,avg_lat_ms,tests,block_number,city,area,typeOfArea,region,digital_elevation_model,last_modified_date,year,quarter,nearest_tower_distance_km
40015,2025-10-01,26.093788,50.627747,172.122,14.584,8,1,952,Halat Umm al Bayd,Ra's Zuwayyid,PT,Capital Governorate,1,6/10/2020,2025,4,3.776998
40016,2025-10-01,26.083921,50.627747,242.191,42.364,17,5,952,`Askar,Ra's Zuwayyid,PT,Capital Governorate,1,6/10/2020,2025,4,2.829574
40017,2025-10-01,26.078988,50.627747,220.642,25.543,24,3,952,`Askar,Ra's Abu Jarjur,PT,Capital Governorate,3,1/18/2012,2025,4,2.404074
40018,2025-10-01,26.014829,50.627747,877.434,77.382,9,1,957,Jaww,Jaww,PPL,Southern Governorate,6,1/18/2012,2025,4,2.609265
40019,2025-10-01,26.009893,50.627747,143.202,18.075,9,1,957,Jaww,Jaww,PPL,Southern Governorate,6,1/18/2012,2025,4,2.240248
40020,2025-10-01,25.728158,50.798035,24.987,12.808,20,3,999,Jazirat `Ajirah,Jazirat `Ajirah,ISL,Southern Governorate,4,6/10/2020,2025,4,35.392960
40021,2025-10-01,25.703413,50.776062,103.508,25.592,21,6,999,Al Hayiya Island,Al Hayiya Island,ISL,Southern Governorate,6,8/3/2016,2025,4,36.705481
40022,2025-10-01,25.698463,50.770569,57.552,43.786,19,1,999,Al Hayiya Island,Al Hayiya Island,ISL,Southern Governorate,6,8/3/2016,2025,4,36.954691
40023,2025-10-01,25.648954,50.737610,190.958,12.248,21,1,999,Jazirat Hawar,Jazirat Hawar,ISL,Southern Governorate,3,6/10/2020,2025,4,40.831303
40024,2025-10-01,25.648954,50.743103,7.973,6.755,163,1,999,Jazirat Hawar,Jazirat Hawar,ISL,Southern Governorate,3,6/10/2020,2025,4,41.006752


In [44]:
speed["nearest_tower_distance_km"]<=1

0        False
1        False
2        False
3        False
4        False
         ...  
40020    False
40021    False
40022    False
40023    False
40024    False
Name: nearest_tower_distance_km, Length: 40025, dtype: bool

In [72]:
#radiance = km / earth radius
# km = radiance * earth radius

for radius_km in [1, 2, 5]:
    r = radius_km / EARTH_RADIUS_KM
    idxs = tree.query_radius(speed_coords, r=r)
    speed[f"tower_count_{radius_km}km"] = [len(x) for x in idxs]

In [73]:
speed["tower_count_5km"]

0        320
1         88
2        138
3        446
4        588
        ... 
40020      0
40021      0
40022      0
40023      0
40024      0
Name: tower_count_5km, Length: 40025, dtype: int64

In [67]:
speed["tower_count_2km"]

0        0
1        2
2        2
3        7
4        0
        ..
40020    0
40021    0
40022    0
40023    0
40024    0
Name: tower_count_2km, Length: 40025, dtype: int64

In [68]:
speed["tower_count_1km"]

0        0
1        0
2        0
3        0
4        0
        ..
40020    0
40021    0
40022    0
40023    0
40024    0
Name: tower_count_1km, Length: 40025, dtype: int64

In [71]:
speed.head()

,date,latitude,longitude,avg_d_mbps,avg_u_mbps,avg_lat_ms,tests,block_number,city,area,...,digital_elevation_model,last_modified_date,year,quarter,nearest_tower_distance_km,tower_count_1km,tower_count_2km,tower_count_5km,tower_count_3km,tower_count_4km
0,2020-01-01,26.286027,50.578308,4.029,3.417,22,1,228,Al Busaytin,Jazirat as Sayah,...,-9999,6/10/2020,2020,1,3.129938,0,0,320,0,50
1,2020-01-01,26.310651,50.622253,11.932,3.027,22,1,263,Khasaifa Island,Khasaifa Island,...,-9999,7/9/2020,2020,1,1.869366,0,2,88,9,20
2,2020-01-01,26.305726,50.622253,32.232,17.256,28,87,263,Khasaifa Island,Khasaifa Island,...,-9999,7/9/2020,2020,1,1.659299,0,2,138,12,39
3,2020-01-01,26.286027,50.600281,23.765,17.882,18,2,228,Al Busaytin,Jazirat as Sayah,...,-9999,6/10/2020,2020,1,1.859964,0,7,446,104,215
4,2020-01-01,26.281102,50.583801,29.366,4.510,22,1,228,Al Busaytin,Jazirat as Sayah,...,-9999,6/10/2020,2020,1,2.368918,0,0,588,30,194


In [80]:
#operator counts within 5 km
r5 = 5 / EARTH_RADIUS_KM
idx_5km = tree.query_radius(speed_coords, r=r5)

In [81]:
speed.head(5)

,date,latitude,longitude,avg_d_mbps,avg_u_mbps,avg_lat_ms,tests,block_number,city,area,...,quarter,nearest_tower_distance_km,tower_count_1km,tower_count_2km,tower_count_5km,tower_count_3km,tower_count_4km,tower_count_5km_zain,tower_count_5km_stc,tower_count_5km_batelco
0,2020-01-01,26.286027,50.578308,4.029,3.417,22,1,228,Al Busaytin,Jazirat as Sayah,...,1,3.129938,0,0,320,0,50,18,0,302
1,2020-01-01,26.310651,50.622253,11.932,3.027,22,1,263,Khasaifa Island,Khasaifa Island,...,1,1.869366,0,2,88,9,20,12,0,76
2,2020-01-01,26.305726,50.622253,32.232,17.256,28,87,263,Khasaifa Island,Khasaifa Island,...,1,1.659299,0,2,138,12,39,16,0,122
3,2020-01-01,26.286027,50.600281,23.765,17.882,18,2,228,Al Busaytin,Jazirat as Sayah,...,1,1.859964,0,7,446,104,215,33,0,413
4,2020-01-01,26.281102,50.583801,29.366,4.510,22,1,228,Al Busaytin,Jazirat as Sayah,...,1,2.368918,0,0,588,30,194,30,0,558


In [88]:
speed.drop(columns=['tower_count_5km_batelco'], inplace=True)

In [79]:
towers.columns

Index(['Site ID', 'Region ID', 'RAT Type', 'RAT SubType', 'Latitude',
       'Longitude', 'Bands', 'Channels', 'First Seen', 'Last Seen', 'Visible',
       'Tower Type', 'Operator'],
      dtype='object')

In [89]:
speed.columns

Index(['date', 'latitude', 'longitude', 'avg_d_mbps', 'avg_u_mbps',
       'avg_lat_ms', 'tests', 'block_number', 'city', 'area', 'typeOfArea',
       'region', 'digital_elevation_model', 'last_modified_date', 'year',
       'quarter', 'nearest_tower_distance_km', 'tower_count_1km',
       'tower_count_2km', 'tower_count_5km', 'tower_count_3km',
       'tower_count_4km'],
      dtype='object')